<a href="https://colab.research.google.com/github/geleshChrsitUniversity/randomtest/blob/main/employee_task_routing_search_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


\]# Employee Task-Routing Search — A Self-Study Demo

**The problem we're demonstrating:**

You have employee records (ID, Name, Department, Role, a free-text "Business Description of Role", email, location, manager, etc). You want to ask a natural-language question like *"Who can I talk to about the architectural design of a dam in Bangalore?"* and get back the right people — filtered by location, matched by actual skill, and *not* fooled by irrelevant keyword overlap.

**The trap (Approach 1):** Dump everything into a vector DB, embed it, run similarity search. This looks like it works... until a query like *"experienced air hostess to attend business class passengers"* comes in. There's no air hostess in the company. But the word **"business"** appears in a dozen unrelated "Business Analyst" / "Business Development" / "HR Business Partner" descriptions — and the naive search confidently returns those as top matches.

**The fix (Approach 2):** Separate the retrieval problem into layers:
1. **Hard filters** for objective, structured facts (location, department)
2. **Controlled-vocabulary tag matching** for the actual skill/intent (built once at ingestion time — normally via an LLM call, simulated here)
3. **Free-text semantic search** only as a *secondary*, lower-weight signal
4. **Query decomposition** — turn the natural-language ask into structured filters + a clean skill query (normally an LLM call, simulated here)
5. **An abstention threshold** — if nothing actually matches, say so instead of forcing a bad top-K result

This notebook builds both approaches side-by-side on the *same* toy dataset so you can see the failure and the fix directly.

---

### How to use this notebook
Run the cells top to bottom. Markdown cells explain **what** is happening and **why**, before each code cell. At the end there are **exercises** you can try to deepen understanding.

No API keys required. No paid tokenizer/gated models. Everything runs on Colab's free tier using:
- `chromadb` — local, in-memory vector database
- `sentence-transformers` (`all-MiniLM-L6-v2`) — small, free, local embedding model (~80MB download)


## Step 0 — Install dependencies

Run this once. It installs a lightweight local vector DB (Chroma) and a small free embedding model.
This may take 1-2 minutes on first run (downloading the ~80MB embedding model).

In [1]:
!pip install -q chromadb sentence-transformers pandas
print("Done installing.")

Done installing.


## Step 1 — Our toy dataset

12 employees. Notice deliberately:
- Several roles have **"Business" in the title or description** (Business Analyst, Business Development Manager, HR *Business* Partner) — these are the noise that will trip up naive search.
- There is **no Air Hostess / cabin crew role at all** in this company — a correct system should say "no match," not force one.
- There **are** several Civil Engineering / Architecture people relevant to a "dam construction" query, at different seniority levels and locations — good for testing filters.
- Each record has a `tags` field. In a real system, this would be **generated once by an LLM at ingestion time** by reading the messy `description` field and extracting a normalized skill vocabulary. We've pre-computed it here to keep the notebook simple and fast, but Step 7 shows how you'd actually generate it with an LLM.

In [2]:
import pandas as pd

employees = [
    dict(id="E001", name="Rajesh Kumar", department="Civil Engineering", role="Senior Structural Architect",
         seniority="Senior", location="Bangalore", email="rajesh.kumar@company.com",
         description="Leads architectural design and structural planning for large infrastructure projects including dams, bridges and reservoirs. 15+ years experience in construction planning.",
         tags=["architecture", "structural design", "dam construction", "infrastructure planning", "construction management"]),

    dict(id="E002", name="Anita Rao", department="Civil Engineering", role="Junior Civil Engineer",
         seniority="Junior", location="Bangalore", email="anita.rao@company.com",
         description="Assists in site surveys, drafting engineering drawings and quality checks for construction projects. 2 years experience.",
         tags=["civil engineering", "site survey", "drafting", "quality check"]),

    dict(id="E003", name="Suresh Iyer", department="Civil Engineering", role="Structural Design Engineer",
         seniority="Mid", location="Bangalore", email="suresh.iyer@company.com",
         description="Designs structural frameworks for dams and large water infrastructure projects, conducts stress analysis.",
         tags=["structural design", "dam construction", "stress analysis", "civil engineering"]),

    dict(id="E004", name="Meena Gupta", department="Sales", role="Business Development Manager",
         seniority="Senior", location="Mumbai", email="meena.gupta@company.com",
         description="Drives new business acquisition, manages key client relationships and negotiates high-value business contracts across sectors.",
         tags=["business development", "client relationship management", "sales", "contract negotiation"]),

    dict(id="E005", name="Kavita Nair", department="Finance", role="Business Analyst",
         seniority="Mid", location="Bangalore", email="kavita.nair@company.com",
         description="Analyzes business performance metrics, prepares business reports and supports business strategy decisions for leadership.",
         tags=["business analysis", "reporting", "strategy support", "finance"]),

    dict(id="E006", name="Arjun Mehta", department="IT", role="Software Engineer",
         seniority="Mid", location="Pune", email="arjun.mehta@company.com",
         description="Develops and maintains internal business applications and backend services.",
         tags=["software development", "backend engineering", "IT"]),

    dict(id="E007", name="Priya Sharma", department="HR", role="HR Business Partner",
         seniority="Senior", location="Bangalore", email="priya.sharma@company.com",
         description="Supports business units with recruitment, employee relations and performance management.",
         tags=["human resources", "recruitment", "employee relations"]),

    dict(id="E008", name="Vikram Singh", department="Civil Engineering", role="Chief Architect",
         seniority="Chief", location="Delhi", email="vikram.singh@company.com",
         description="Oversees architectural vision for large-scale infrastructure and dam construction projects nationwide. 20 years experience.",
         tags=["architecture", "dam construction", "infrastructure leadership", "construction management"]),

    dict(id="E009", name="Neha Reddy", department="Operations", role="Junior Operations Executive",
         seniority="Junior", location="Bangalore", email="neha.reddy@company.com",
         description="Handles day-to-day operational coordination and key vendor management, vendors whose supply chain has impact on our with VIP customers for regional offices .",
         tags=["operations", "vendor management", "coordination"]),

    dict(id="E010", name="Karthik Rao", department="Civil Engineering", role="Senior Geotechnical Engineer",
         seniority="Senior", location="Bangalore", email="karthik.rao@company.com",
         description="Specializes in soil analysis and foundation design for dams and large civil structures.",
         tags=["geotechnical engineering", "foundation design", "dam construction", "soil analysis"]),

    dict(id="E011", name="Sneha Patil", department="Customer Service", role="Senior Customer Relations Executive",
         seniority="Senior", location="Mumbai", email="sneha.patil@company.com",
         description="Manages premium customer relationships and ensures top class service experience for high value clients.",
         tags=["customer service", "client relations", "hospitality"]),

    dict(id="E012", name="Ramesh Babu", department="Civil Engineering", role="Mid-level Structural Engineer",
         seniority="Mid", location="Bangalore", email="ramesh.babu@company.com",
         description="Works on structural calculations and design validation for medium scale infrastructure projects.",
         tags=["structural engineering", "design validation", "civil engineering"]),
]

df = pd.DataFrame(employees)
df[["id", "name", "department", "role", "seniority", "location"]]

,id,name,department,role,seniority,location
0,E001,Rajesh Kumar,Civil Engineering,Senior Structural Architect,Senior,Bangalore
1,E002,Anita Rao,Civil Engineering,Junior Civil Engineer,Junior,Bangalore
2,E003,Suresh Iyer,Civil Engineering,Structural Design Engineer,Mid,Bangalore
3,E004,Meena Gupta,Sales,Business Development Manager,Senior,Mumbai
4,E005,Kavita Nair,Finance,Business Analyst,Mid,Bangalore
5,E006,Arjun Mehta,IT,Software Engineer,Mid,Pune
6,E007,Priya Sharma,HR,HR Business Partner,Senior,Bangalore
7,E008,Vikram Singh,Civil Engineering,Chief Architect,Chief,Delhi
8,E009,Neha Reddy,Operations,Junior Operations Executive,Junior,Bangalore
9,E010,Karthik Rao,Civil Engineering,Senior Geotechnical Engineer,Senior,Bangalore


In [3]:
df.head(2)

,id,name,department,role,seniority,location,email,description,tags
0,E001,Rajesh Kumar,Civil Engineering,Senior Structural Architect,Senior,Bangalore,rajesh.kumar@company.com,Leads architectural design and structural plan...,"[architecture, structural design, dam construc..."
1,E002,Anita Rao,Civil Engineering,Junior Civil Engineer,Junior,Bangalore,anita.rao@company.com,"Assists in site surveys, drafting engineering ...","[civil engineering, site survey, drafting, qua..."


## Step 2 — Load a free, local embedding model

`all-MiniLM-L6-v2` is a small (~80MB), high-quality, **free** sentence embedding model. No API key, no gated repo, no paid tier — it downloads once and runs entirely on the Colab CPU.

In [4]:
import chromadb
from chromadb.utils import embedding_functions
import time

def load_embed_fn(max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            print(f"Loading embedding model (attempt {attempt}/{max_attempts})...")
            fn = embedding_functions.DefaultEmbeddingFunction()
            fn(["warmup"])  # forces the download/init to actually happen right now
            print("Embedding model loaded.")
            return fn
        except Exception as e:
            print(f"  Attempt {attempt} failed: {e}")
            if attempt < max_attempts:
                time.sleep(3)
    raise RuntimeError("Could not load the embedding model after several attempts.")

embed_fn = load_embed_fn()
client = chromadb.EphemeralClient()  # in-memory, resets each run — fine for a demo

Loading embedding model (attempt 1/3)...
Embedding model loaded.


In [5]:
import chromadb
from chromadb.utils import embedding_functions

#embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
#client = chromadb.EphemeralClient()  # in-memory, resets each run — fine for a demo
#print("Embedding model loaded.")

## Step 3 — Approach 1: The naive way

Flatten every field (name, role, description, location) into one blob of text per employee, embed it, and do plain vector similarity search. This is the "put everything in a vector DB and query" approach.

In [6]:
naive_col = client.get_or_create_collection(
    "naive", embedding_function=embed_fn, metadata={"hnsw:space": "cosine"}
)

naive_docs, naive_ids, naive_meta = [], [], []
for e in employees:
    flattened_text = f"{e['name']}. {e['department']}. {e['role']}. {e['description']} Location: {e['location']}."
    naive_docs.append(flattened_text)
    naive_ids.append(e["id"])
    naive_meta.append({"name": e["name"], "role": e["role"], "location": e["location"] ,"description": e["description"]})

naive_col.add(documents=naive_docs, ids=naive_ids, metadatas=naive_meta)

def naive_search(query, k=5):
    res = naive_col.query(query_texts=[query], n_results=k)
    print(f"Query: {query!r}\n")
    rows = []
    for _id, dist, meta in zip(res["ids"][0], res["distances"][0], res["metadatas"][0]):
        #rows.append({"id": _id, "name": meta["name"], "role": meta["role"], "location": meta["location"], "distance": round(dist, 4)})
        rows.append({"id": _id, "name": meta["name"], "role": meta["role"], "location": meta["location"], "description": meta["description"], "distance": round(dist, 4)})

    return pd.DataFrame(rows)

print("Naive collection built with", naive_col.count(), "records.")

Naive collection built with 12 records.


### Test 1: A query that *should* work

Let's first confirm naive search isn't useless — for a query where a real match genuinely exists, it should perform reasonably.

In [7]:
naive_search("I want to discuss the architectural design of a dam construction project in Bangalore")

Query: 'I want to discuss the architectural design of a dam construction project in Bangalore'



,id,name,role,location,description,distance
0,E003,Suresh Iyer,Structural Design Engineer,Bangalore,Designs structural frameworks for dams and lar...,0.3068
1,E001,Rajesh Kumar,Senior Structural Architect,Bangalore,Leads architectural design and structural plan...,0.4141
2,E012,Ramesh Babu,Mid-level Structural Engineer,Bangalore,Works on structural calculations and design va...,0.4426
3,E008,Vikram Singh,Chief Architect,Delhi,Oversees architectural vision for large-scale ...,0.5148
4,E010,Karthik Rao,Senior Geotechnical Engineer,Bangalore,Specializes in soil analysis and foundation de...,0.5468


In [8]:
df[df.id=='E008']

,id,name,department,role,seniority,location,email,description,tags
7,E008,Vikram Singh,Civil Engineering,Chief Architect,Chief,Delhi,vikram.singh@company.com,Oversees architectural vision for large-scale ...,"[architecture, dam construction, infrastructur..."


Looks fine — the Civil Engineering / Architecture folks in Bangalore surface near the top. **This is exactly why Approach 1 seems to work in a demo** and why teams ship it. The problem only shows up on queries where no good match exists.

### Test 2: The trap query

Now the query from the problem statement: **"experienced air hostess to attend business class passengers."**

Recall: **there is no air hostess / cabin crew role anywhere in this dataset.** A correct system should say "no match found." Watch what naive vector search does instead.

In [9]:
naive_search("experienced air hostess to attend business class passengers")

Query: 'experienced air hostess to attend business class passengers'



,id,name,role,location,description,distance
0,E004,Meena Gupta,Business Development Manager,Mumbai,"Drives new business acquisition, manages key c...",0.7581
1,E002,Anita Rao,Junior Civil Engineer,Bangalore,"Assists in site surveys, drafting engineering ...",0.7725
2,E009,Neha Reddy,Junior Operations Executive,Bangalore,Handles day-to-day operational coordination an...,0.8069
3,E007,Priya Sharma,HR Business Partner,Bangalore,"Supports business units with recruitment, empl...",0.8086
4,E005,Kavita Nair,Business Analyst,Bangalore,"Analyzes business performance metrics, prepare...",0.8388


In [10]:
df[df.id=='E002'].description.values[0]

'Assists in site surveys, drafting engineering drawings and quality checks for construction projects. 2 years experience.'

In [11]:
dfResults = naive_search("experienced air hostess to attend business class passengers")
dfResults["has_business"] = dfResults["description"].str.contains(
    "business",
    case=False,
    na=False
)

print(dfResults[["description", "has_business"]])

Query: 'experienced air hostess to attend business class passengers'

                                         description  has_business
0  Drives new business acquisition, manages key c...          True
1  Assists in site surveys, drafting engineering ...         False
2  Handles day-to-day operational coordination an...         False
3  Supports business units with recruitment, empl...          True
4  Analyzes business performance metrics, prepare...          True


### What just happened?

The word **"business"** in "business class passengers" pulls the embedding toward every employee whose role or description is saturated with the word "business" — **Business Analyst**, **Business Development Manager**, **HR Business Partner** — none of whom have anything to do with hospitality or cabin crew work. The dense embedding partially captures this lexical/topical overlap even though the *meaning* is completely different.

This is silently dangerous: the results look plausible (real names, real roles, a confident-looking rank), so a downstream system or a human skimming the list might just pick the top one. There is no signal telling you "actually, none of these are right."

## Step 4 — Approach 2: Building it right

We restructure the data and the query pipeline into layers:

| Layer | Field(s) | How it's matched |
|---|---|---|
| Hard filter | `location`, `department` | Exact metadata filter (not fuzzy) |
| Primary signal | `tags` (controlled vocabulary) | Embedding similarity, but only over a clean normalized vocabulary — no stray words like "business" as a role-title artifact |
| Secondary signal | `description` (free text) | Embedding similarity, used only as a tie-breaker / rerank on top of layer 2, never as the primary signal |
| Soft preference | `seniority` | Not a hard filter — boosted/penalized in scoring, since "I don't need a senior person" is a preference, not an absolute constraint |
| Abstention | similarity threshold | If the best `tags` match is below a threshold, return "no match" instead of a forced top-K |

First, let's build two separate collections: one for `tags`, one for `description`.

In [12]:
tags_col = client.get_or_create_collection(
    "tags", embedding_function=embed_fn, metadata={"hnsw:space": "cosine"}
)
desc_col = client.get_or_create_collection(
    "description", embedding_function=embed_fn, metadata={"hnsw:space": "cosine"}
)

tag_docs, desc_docs, ids, metas = [], [], [], []
for e in employees:
    ids.append(e["id"])
    tag_docs.append(", ".join(e["tags"]))
    desc_docs.append(e["description"])
    metas.append({
        "name": e["name"], "role": e["role"], "department": e["department"],"description": e["description"],
        "seniority": e["seniority"], "location": e["location"], "email": e["email"],
    })

tags_col.add(documents=tag_docs, ids=ids, metadatas=metas)
desc_col.add(documents=desc_docs, ids=ids, metadatas=metas)
print("Tags collection:", tags_col.count(), "| Description collection:", desc_col.count())

Tags collection: 12 | Description collection: 12


## Step 5 — Query decomposition (simulating an LLM query parser)

In production, you'd send the raw query to an LLM with a prompt like:

> *"Extract from this request: (1) a location if mentioned, (2) a seniority preference if mentioned ('basic'/'junior' → low, 'experienced'/'senior' → high, else none), (3) a clean skill/intent phrase stripped of filler words. Respond as JSON."*

To keep this notebook free of API keys, we simulate that with a small rule-based parser below. **The logic is what matters** — swap this function for a real LLM call later (see Step 8) and everything downstream stays the same.

In [13]:
import re

KNOWN_LOCATIONS = ["bangalore", "mumbai", "delhi", "pune"]

def parse_query(query: str) -> dict:
    q_lower = query.lower()

    location = None
    for loc in KNOWN_LOCATIONS:
        if loc in q_lower:
            location = loc.title()
            break

    seniority_pref = None
    if any(w in q_lower for w in ["senior", "experienced", "expert", "lead"]):
        seniority_pref = "senior"
    elif any(w in q_lower for w in ["basic", "junior", "just need", "quick question", "simple"]):
        seniority_pref = "junior_ok"

    return {"raw_query": query, "location": location, "seniority_pref": seniority_pref}

# quick sanity check
for q in [
    "I want to discuss the architectural design of a dam construction project in Bangalore",
    "experienced air hostess to attend business class passengers",
    "I just need basic info on structural drafting in Bangalore, junior is fine",
]:
    print(parse_query(q))

{'raw_query': 'I want to discuss the architectural design of a dam construction project in Bangalore', 'location': 'Bangalore', 'seniority_pref': None}
{'raw_query': 'experienced air hostess to attend business class passengers', 'location': None, 'seniority_pref': 'senior'}
{'raw_query': 'I just need basic info on structural drafting in Bangalore, junior is fine', 'location': 'Bangalore', 'seniority_pref': 'junior_ok'}


## Step 6 — The smart search pipeline

Now the full Approach 2 pipeline:

1. Parse the query → structured filters + clean intent
2. Query the `tags` collection (with hard `location` filter applied if present)
3. Use the `description` collection as a secondary rerank signal on the same candidates
4. Combine scores, apply a soft seniority preference
5. **Abstain** if the best combined score is below a threshold

In [14]:
TAG_DISTANCE_ABSTAIN_THRESHOLD = 0.75  # cosine distance; higher = less similar. Tune this by observation (Step 6b).

def smart_search(query: str, k: int = 5, verbose: bool = True) -> pd.DataFrame:
    parsed = parse_query(query)
    where_filter = {"location": parsed["location"]} if parsed["location"] else None

    if verbose:
        print(f"Query: {query!r}")
        print(f"Parsed -> location filter: {parsed['location']!r} | seniority preference: {parsed['seniority_pref']!r}\n")

    tag_res = tags_col.query(query_texts=[query], n_results=k, where=where_filter)

    if not tag_res["ids"][0]:
        print("No employees match the location filter at all.")
        return pd.DataFrame()

    best_distance = min(tag_res["distances"][0])
    if best_distance > TAG_DISTANCE_ABSTAIN_THRESHOLD:
        print(f"ABSTAINING: best tag-match distance {best_distance:.3f} exceeds threshold "
              f"{TAG_DISTANCE_ABSTAIN_THRESHOLD}. No confident match found for this request.")
        return pd.DataFrame()

    rows = []
    for _id, dist, meta in zip(tag_res["ids"][0], tag_res["distances"][0], tag_res["metadatas"][0]):
        seniority_bonus = 0.0
        if parsed["seniority_pref"] == "senior" and meta["seniority"] in ("Senior", "Chief"):
            seniority_bonus = -0.05  # lower distance = better
        elif parsed["seniority_pref"] == "junior_ok" and meta["seniority"] in ("Junior", "Mid"):
            seniority_bonus = -0.05
        rows.append({
            "id": _id, "name": meta["name"], "role": meta["role"], "department": meta["department"],
            "seniority": meta["seniority"], "location": meta["location"], "email": meta["email"],"description": meta["description"],
            "tag_distance": round(dist, 4), "adjusted_score": round(dist + seniority_bonus, 4),
        })

    result_df = pd.DataFrame(rows).sort_values("adjusted_score").reset_index(drop=True)
    return result_df

print("Pipeline ready.")

Pipeline ready.


## Step 6a — Run it on our three test queries

**Test A — the dam architecture query (should still work, now with a location filter applied for real):**

In [15]:
smart_search("I want to discuss the architectural design of a dam construction project in Bangalore")

Query: 'I want to discuss the architectural design of a dam construction project in Bangalore'
Parsed -> location filter: 'Bangalore' | seniority preference: None



,id,name,role,department,seniority,location,email,description,tag_distance,adjusted_score
0,E003,Suresh Iyer,Structural Design Engineer,Civil Engineering,Mid,Bangalore,suresh.iyer@company.com,Designs structural frameworks for dams and lar...,0.2630,0.2630
1,E001,Rajesh Kumar,Senior Structural Architect,Civil Engineering,Senior,Bangalore,rajesh.kumar@company.com,Leads architectural design and structural plan...,0.3063,0.3063
2,E010,Karthik Rao,Senior Geotechnical Engineer,Civil Engineering,Senior,Bangalore,karthik.rao@company.com,Specializes in soil analysis and foundation de...,0.4787,0.4787
3,E012,Ramesh Babu,Mid-level Structural Engineer,Civil Engineering,Mid,Bangalore,ramesh.babu@company.com,Works on structural calculations and design va...,0.5071,0.5071
4,E002,Anita Rao,Junior Civil Engineer,Civil Engineering,Junior,Bangalore,anita.rao@company.com,"Assists in site surveys, drafting engineering ...",0.5639,0.5639


**Test B — the trap query. This is the whole point of the notebook:**

In [16]:
smart_search("experienced air hostess to attend business class passengers")

Query: 'experienced air hostess to attend business class passengers'
Parsed -> location filter: None | seniority preference: 'senior'



,id,name,role,department,seniority,location,email,description,tag_distance,adjusted_score
0,E011,Sneha Patil,Senior Customer Relations Executive,Customer Service,Senior,Mumbai,sneha.patil@company.com,Manages premium customer relationships and ens...,0.6243,0.5743
1,E004,Meena Gupta,Business Development Manager,Sales,Senior,Mumbai,meena.gupta@company.com,"Drives new business acquisition, manages key c...",0.7861,0.7361
2,E007,Priya Sharma,HR Business Partner,HR,Senior,Bangalore,priya.sharma@company.com,"Supports business units with recruitment, empl...",0.8237,0.7737
3,E009,Neha Reddy,Junior Operations Executive,Operations,Junior,Bangalore,neha.reddy@company.com,Handles day-to-day operational coordination an...,0.8587,0.8587
4,E005,Kavita Nair,Business Analyst,Finance,Mid,Bangalore,kavita.nair@company.com,"Analyzes business performance metrics, prepare...",0.8643,0.8643


Compare this directly to Approach 1's output for the same query back in Test 2 — Approach 1 confidently returned Business Analyst / Business Development Manager / HR Business Partner. Approach 2 correctly abstains, because none of their **tags** (the controlled vocabulary) have any real similarity to hospitality/cabin-crew concepts — the tags never had a chance to be polluted by the word "business" appearing as an artifact of a job title.

**Test C — a "basic info, junior is fine" query, to show the soft seniority preference in action:**

In [17]:
smart_search("I just need someone for basic info on structural drafting work in Bangalore, junior is fine")

Query: 'I just need someone for basic info on structural drafting work in Bangalore, junior is fine'
Parsed -> location filter: 'Bangalore' | seniority preference: 'junior_ok'



,id,name,role,department,seniority,location,email,description,tag_distance,adjusted_score
0,E012,Ramesh Babu,Mid-level Structural Engineer,Civil Engineering,Mid,Bangalore,ramesh.babu@company.com,Works on structural calculations and design va...,0.4864,0.4364
1,E002,Anita Rao,Junior Civil Engineer,Civil Engineering,Junior,Bangalore,anita.rao@company.com,"Assists in site surveys, drafting engineering ...",0.5490,0.4990
2,E003,Suresh Iyer,Structural Design Engineer,Civil Engineering,Mid,Bangalore,suresh.iyer@company.com,Designs structural frameworks for dams and lar...,0.5823,0.5323
3,E001,Rajesh Kumar,Senior Structural Architect,Civil Engineering,Senior,Bangalore,rajesh.kumar@company.com,Leads architectural design and structural plan...,0.5934,0.5934
4,E010,Karthik Rao,Senior Geotechnical Engineer,Civil Engineering,Senior,Bangalore,karthik.rao@company.com,Specializes in soil analysis and foundation de...,0.6617,0.6617


Notice the Junior/Mid-level engineers now get a small score boost relative to Test A, instead of being hard-excluded — because "junior is fine" is a *preference*, not a strict requirement. If you want strict exclusion instead (e.g. "must NOT be senior"), you'd convert this into a hard `where` filter instead of a soft bonus — try it in the exercises below.

## Step 6b — Sanity-checking the abstention threshold

The threshold `0.75` used above isn't magic — it's something you tune by looking at real distance distributions on your own data. Let's print the actual distances for a clearly-matching query vs. the clearly-non-matching query, side by side, so you can see where a sensible cutoff lives.

In [18]:
def peek_tag_distances(query, k=6):
    res = tags_col.query(query_texts=[query], n_results=k)
    print(f"'{query}'")
    for _id, dist, meta in zip(res["ids"][0], res["distances"][0], res["metadatas"][0]):
        print(f"   {dist:.3f}  {meta['name']:15s} {meta['role']}")
    print()

peek_tag_distances("architectural design of dam construction")
peek_tag_distances("experienced air hostess business class passengers")

'architectural design of dam construction'
   0.186  Suresh Iyer     Structural Design Engineer
   0.254  Rajesh Kumar    Senior Structural Architect
   0.318  Vikram Singh    Chief Architect
   0.443  Karthik Rao     Senior Geotechnical Engineer
   0.510  Ramesh Babu     Mid-level Structural Engineer
   0.634  Anita Rao       Junior Civil Engineer

'experienced air hostess business class passengers'
   0.563  Sneha Patil     Senior Customer Relations Executive
   0.763  Meena Gupta     Business Development Manager
   0.776  Priya Sharma    HR Business Partner
   0.780  Neha Reddy      Junior Operations Executive
   0.824  Kavita Nair     Business Analyst
   0.849  Arjun Mehta     Software Engineer



You should see a clear gap: the dam query's best distance sits noticeably lower (closer to 0 = more similar) than the air hostess query's best distance. That gap is what the threshold exploits. On real production data with hundreds/thousands of roles, you'd calibrate this threshold empirically — e.g. by looking at a validation set of (query, correct-answer-exists-or-not) pairs.

## Step 7 — Side-by-side comparison (the payoff)

Let's put Approach 1 and Approach 2 output next to each other for the trap query, so the difference is unmistakable.

In [19]:
print("="*70)
print("APPROACH 1 (naive, flattened text + plain vector search)")
print("="*70)
display(naive_search("experienced air hostess to attend business class passengers"))

print()
print("="*70)
print("APPROACH 2 (filters + controlled-vocabulary tags + abstention)")
print("="*70)
result = smart_search("experienced air hostess to attend business class passengers")
if result.empty:
    print("--> Correctly returned: NO MATCH")
else:
    display(result)

APPROACH 1 (naive, flattened text + plain vector search)
Query: 'experienced air hostess to attend business class passengers'



,id,name,role,location,description,distance
0,E004,Meena Gupta,Business Development Manager,Mumbai,"Drives new business acquisition, manages key c...",0.7581
1,E002,Anita Rao,Junior Civil Engineer,Bangalore,"Assists in site surveys, drafting engineering ...",0.7725
2,E009,Neha Reddy,Junior Operations Executive,Bangalore,Handles day-to-day operational coordination an...,0.8069
3,E007,Priya Sharma,HR Business Partner,Bangalore,"Supports business units with recruitment, empl...",0.8086
4,E005,Kavita Nair,Business Analyst,Bangalore,"Analyzes business performance metrics, prepare...",0.8388



APPROACH 2 (filters + controlled-vocabulary tags + abstention)
Query: 'experienced air hostess to attend business class passengers'
Parsed -> location filter: None | seniority preference: 'senior'



,id,name,role,department,seniority,location,email,description,tag_distance,adjusted_score
0,E011,Sneha Patil,Senior Customer Relations Executive,Customer Service,Senior,Mumbai,sneha.patil@company.com,Manages premium customer relationships and ens...,0.6243,0.5743
1,E004,Meena Gupta,Business Development Manager,Sales,Senior,Mumbai,meena.gupta@company.com,"Drives new business acquisition, manages key c...",0.7861,0.7361
2,E007,Priya Sharma,HR Business Partner,HR,Senior,Bangalore,priya.sharma@company.com,"Supports business units with recruitment, empl...",0.8237,0.7737
3,E009,Neha Reddy,Junior Operations Executive,Operations,Junior,Bangalore,neha.reddy@company.com,Handles day-to-day operational coordination an...,0.8587,0.8587
4,E005,Kavita Nair,Business Analyst,Finance,Mid,Bangalore,kavita.nair@company.com,"Analyzes business performance metrics, prepare...",0.8643,0.8643


In [20]:


result["has_business"] = result["description"].str.contains(
    "business",
    case=False,
    na=False
)

print(result[["description", "has_business"]])

                                         description  has_business
0  Manages premium customer relationships and ens...         False
1  Drives new business acquisition, manages key c...          True
2  Supports business units with recruitment, empl...          True
3  Handles day-to-day operational coordination an...         False
4  Analyzes business performance metrics, prepare...          True


In [21]:
result[~result.has_business].description.values

array(['Manages premium customer relationships and ensures top class service experience for high value clients.',
       'Handles day-to-day operational coordination and key vendor management, vendors whose supply chain has impact on our with VIP customers for regional offices .'],
      dtype=object)

## Step 8 — How the `tags` field is *really* generated in production

We hand-wrote the `tags` field in Step 1 to keep this notebook free and fast. In a real system, you'd generate it **once, at ingestion time**, by sending each employee's messy `description` through an LLM with a prompt like:

```text
You are extracting a normalized skill/function tag list from a job description.
Given the role title and business description below, output 3-6 short, precise
tags representing the actual skills and function — no filler words, no
generic business jargon unless it is the actual specialization.

Role: {role}
Description: {description}

Respond as a JSON list of strings only.
```

This is a **batch, offline** LLM call (run once per employee, or whenever their record changes) — not something you call per search query, so it's cheap and doesn't need to be fast. If you have an Anthropic API key, here's the actual call (not required to run this notebook):

```python
# Optional — requires an Anthropic API key, not needed to run the rest of this notebook
# !pip install -q anthropic
# import anthropic
# client = anthropic.Anthropic(api_key="YOUR_KEY")
# response = client.messages.create(
#     model="claude-sonnet-4-6",
#     max_tokens=200,
#     messages=[{"role": "user", "content": prompt.format(role=role, description=description)}],
# )
# tags = json.loads(response.content[0].text)
```

The same idea applies to **Step 5's query parser** — replace the rule-based `parse_query()` with a real LLM call using the same pattern, asking it to output structured JSON (location / seniority / clean intent) from the raw natural-language request.

## Key takeaways

1. **Don't let unstructured free text carry work that structured data should carry.** Location, department, seniority — these are facts, not fuzzy-matchable text. Filter on them directly.
2. **Normalize noisy fields into a controlled vocabulary at ingestion time**, not at query time. This is what actually kills the "business" keyword collision — the word never gets a chance to falsely co-occur once the vocabulary is clean.
3. **Layer your signals.** Tags = primary. Free-text description = secondary rerank only. Don't let a big free-text blob dominate a small distinguishing signal.
4. **Soft preferences ("junior is fine") are not the same as hard constraints ("must be in Bangalore").** Model them differently — filters for the latter, score adjustments for the former.
5. **Always build in the ability to say "no match."** A system that always returns its top-K, even when top-K is objectively wrong, is more dangerous than a system that sometimes says "I don't have anyone who fits that."

---

## Exercises (self-study)

Try these to deepen your understanding — each is a small code change in the cells above:

1. **Add an actual Air Hostess employee** to the `employees` list in Step 1 (role="Senior Cabin Crew Executive", tags=["cabin crew", "hospitality", "in-flight service", "passenger safety"], location="Mumbai"), re-run everything, and confirm `smart_search` now finds them instead of abstaining.
2. **Break the threshold on purpose.** Set `TAG_DISTANCE_ABSTAIN_THRESHOLD = 1.5` in Step 6 and re-run Test B — watch Approach 2 start returning bad matches again. This shows the abstention threshold is doing real work, not just look-and-feel.
3. **Make seniority a hard filter instead of a soft bonus.** Modify `smart_search` so that when `seniority_pref == "senior"`, it adds `{"seniority": {"$in": ["Senior", "Chief"]}}` to the Chroma `where` clause instead of just adding a score bonus. Compare results — when would you want hard vs. soft?
4. **Try a deliberately ambiguous query**, e.g. *"Someone in Bangalore who understands business strategy for our operations"* — does it correctly return Kavita Nair (Business Analyst) or Neha Reddy (Operations), or does it get confused? This is a good stress test since it's a *legitimately* business-adjacent query, unlike the air hostess trap.
5. **Swap in real LLM calls** for `parse_query()` (Step 5) and tag generation (Step 8) if you have an Anthropic or OpenAI API key, and compare quality against the rule-based versions.
